In [0]:
SELECT DISTINCT PRODUCT_NAME, productgroup_name FROM adb_rtp.silver.daily_pricing_silver WHERE PRODUCT_NAME='Onion'

In [0]:
UPDATE adb_rtp.silver.daily_pricing_silver
SET PRODUCTGROUP_NAME='Ground Vegetables',
lakehouse_updated_date = current_timestamp()
WHERE PRODUCT_NAME='Onion'

In [0]:
CREATE TABLE IF NOT EXISTS adb_rtp.gold.reporting_dim_product_gold_SCDTYPE2 (
  PRODUCTGROUP_NAME STRING,
  PRODUCT_NAME STRING,
  PRODUCT_ID BIGINT,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  lakehouse_inserted_date TIMESTAMP,
  lakehouse_updated_date TIMESTAMP)
USING delta

--Start_date and end_date show the date when the original record came in and the end date is the date when the table was updated with new product_group and likewise for the next records

In [0]:
use catalog adb_rtp;
truncate table silver.reporting_dim_product_stage_1;
truncate table silver.reporting_dim_product_stage_2;
truncate table silver.reporting_dim_product_stage_3;
truncate table gold.reporting_dim_product_gold_SCDTYPE2;

In [0]:
CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_1 AS
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM adb_rtp.silver.daily_pricing_silver
WHERE lakehouse_updated_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'2023-05-01') FROM adb_rtp.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoadScdType2' AND process_status = 'Completed' );

In [0]:
CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_2 AS 
SELECT 
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
  ,goldDim.PRODUCT_NAME AS GOLD_PRODUCT_NAME
  ,goldDim.PRODUCT_ID AS GOLD_PRODUCT_ID
,ROW_NUMBER() OVER (  ORDER BY silverDim.PRODUCT_NAME,silverDim.PRODUCTGROUP_NAME)  as PRODUCT_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM adb_rtp.silver.reporting_dim_product_stage_1 silverDim
LEFT OUTER JOIN adb_rtp.gold.reporting_dim_product_gold_SCDTYPE2 goldDim
ON silverDim.PRODUCT_NAME= goldDim.PRODUCT_NAME
AND goldDim.end_date is null
WHERE goldDim.PRODUCT_NAME IS NULL OR silverDim.PRODUCTGROUP_NAME <> goldDim.PRODUCTGROUP_NAME
-- Need to create a new product_id for the new updated rows in SCDTYPE2

In [0]:
CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_3 AS 
SELECT
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
  ,GOLD_PRODUCT_ID
 ,silverDim.PRODUCT_ID + PREV_MAX_SK_ID  as PRODUCT_ID
,CASE WHEN GOLD_PRODUCT_NAME IS NULL THEN 'New' Else 'Changed' End as RECORD_STATUS
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
adb_rtp.silver.reporting_dim_product_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(PRODUCT_ID),0) as PREV_MAX_SK_ID FROM  adb_rtp.gold.reporting_dim_product_gold_SCDTYPE2 ) goldDim;
--New surrogate id for each row, not just new values
--Create a new column Gold_product_name to use for the insert statement where 'New' is for completely new columns and 'Changed' is for

In [0]:
MERGE INTO  adb_rtp.gold.reporting_dim_product_gold_SCDTYPE2 goldDim
USING adb_rtp.silver.reporting_dim_product_stage_3 silverDim
ON goldDim.PRODUCT_ID = silverDim.GOLD_PRODUCT_ID
WHEN MATCHED THEN 
UPDATE SET goldDim.end_date=current_timestamp()
          ,goldDim.lakehouse_updated_date=current_timestamp()
WHEN NOT MATCHED  THEN
INSERT (PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,start_date,end_date,lakehouse_inserted_date,lakehouse_updated_date)
VALUES (silverDim.PRODUCTGROUP_NAME,silverDim.PRODUCT_NAME,silverDim.PRODUCT_ID,current_timestamp(),NULL,current_timestamp(),current_timestamp())

--When the product_id matches to the existing data in gold table we dont make any change to the table. When the product ID is not matching we insert all the data points as new row


In [0]:
INSERT INTO adb_rtp.gold.reporting_dim_product_gold_SCDTYPE2 
SELECT
PRODUCTGROUP_NAME
,PRODUCT_NAME
,PRODUCT_ID
,current_timestamp()
,NULL
,current_timestamp()
,current_timestamp()
FROM adb_rtp.silver.reporting_dim_product_stage_3
WHERE RECORD_STATUS ='Changed'

--We need to write an additional insert statement due to limitations in merge statement where we cannot insert new rows and update the table for the same records, for example in case of onion where we are comparing with previously existing product_id value in both silver and gold tables but a new id is created for the updated product name in staging table

In [0]:
INSERT INTO  adb_rtp.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESSED_TABLE_DATETIME,PROCESS_STATUS)
SELECT 'reportingDimensionTablesLoadScdType2' , max(lakehouse_updated_date) ,'Completed' FROM adb_rtp.silver.daily_pricing_silver